# RTU 전력 예측 모델링
### EDA 결론 기반 피처 전략
- 포함: hour_sin/cos, currentR/S/T, reactivePowerLagging
- 제외: dow, is_weekend, voltage, powerFactor, operation (EDA 근거)
- 방식: 전체 합산 직접 예측 (설비별 패턴 차이 없음 확인)
- 타겟: hourly_pow (5월 672시간)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows: 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
print('라이브러리 로드 완료')

라이브러리 로드 완료


In [3]:
pip install lightgbm statsmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. 데이터 로드 & hourly_full 생성
> EDA 노트북에서 했던 전처리 그대로 재현

In [2]:
# ⚠️ 파일 경로 본인 환경에 맞게 수정
df = pd.read_csv('rtu_data_full.csv')

# 시간 변환
df['dt'] = pd.to_datetime(df['localtime'], format='%Y%m%d%H%M%S')

# 설비별 시간 단위 집계
agg_cols = {
    'activePower': 'mean',
    'currentR': 'mean',
    'currentS': 'mean',
    'currentT': 'mean',
    'reactivePowerLagging': 'mean'
}

df_hourly_module = (
    df.groupby(['module(equipment)', pd.Grouper(key='dt', freq='1H')])
    .agg(agg_cols)
    .reset_index()
)

# 전체 설비 합산
hourly_full = (
    df_hourly_module.groupby('dt')
    .agg({
        'activePower': 'sum',
        'currentR': 'mean',
        'currentS': 'mean',
        'currentT': 'mean',
        'reactivePowerLagging': 'mean'
    })
    .reset_index()
    .rename(columns={'activePower': 'hourly_pow'})
)

# 시간 피처
hourly_full['hour'] = hourly_full['dt'].dt.hour
hourly_full['hour_sin'] = np.sin(2 * np.pi * hourly_full['hour'] / 24)
hourly_full['hour_cos'] = np.cos(2 * np.pi * hourly_full['hour'] / 24)

print('hourly_full shape:', hourly_full.shape)
print(hourly_full.head())

hourly_full shape: (3601, 9)
                   dt    hourly_pow   currentR   currentS   currentT  \
0 2024-12-01 00:00:00  39222.026694  17.545821  17.513784  17.572936   
1 2024-12-01 01:00:00  39229.376486  17.644235  17.504292  17.491845   
2 2024-12-01 02:00:00  38934.010514  17.471162  17.390261  17.376300   
3 2024-12-01 03:00:00  39066.889514  17.499027  17.407326  17.508141   
4 2024-12-01 04:00:00  39149.150819  17.587642  17.416513  17.529465   

   reactivePowerLagging  hour  hour_sin  hour_cos  
0            603.626283     0  0.000000  1.000000  
1            605.658051     1  0.258819  0.965926  
2            600.846719     2  0.500000  0.866025  
3            601.699624     3  0.707107  0.707107  
4            600.023209     4  0.866025  0.500000  


## 2. 학습/테스트 분리
> 시간 기준 분할 — 12월~4월 학습 / 5월 예측

In [ ]:
test_df = pd.read_csv('test.csv')
print(test_df.shape)
print(test_df.head())
print(test_df.columns.tolist())

(6514573, 20)
  module(equipment)      timestamp       localtime  operation  voltageR  \
0           1(PM-3)  1743490800000  20250401000000          1    218.91   
1           1(PM-3)  1743490805000  20250401000005          1    219.60   
2           1(PM-3)  1743490810000  20250401000010          1    212.16   
3           1(PM-3)  1743490815000  20250401000015          1    217.85   
4           1(PM-3)  1743490820000  20250401000020          1    218.86   

   voltageS  voltageT  voltageRS  voltageST  voltageTR  currentR  currentS  \
0    213.07    218.34     374.09     373.60     378.66      9.83      5.39   
1    210.74    218.75     372.68     371.94     379.61     22.86     24.05   
2    217.28    214.57     371.89     373.99     369.55     18.61     15.73   
3    214.18    216.50     374.13     372.97     376.14     17.86      6.31   
4    219.11    211.99     379.28     373.32     373.11     22.80     27.80   

   currentT  activePower  powerFactorR  powerFactorS  powerFactorT

In [5]:
test_df['datetime'] = pd.to_datetime(test_df['datetime'])
print('test 기간:', test_df['datetime'].min(), '~', test_df['datetime'].max())

test 기간: 2025-04-01 00:00:00 ~ 2025-04-30 00:00:00


In [6]:
sample = pd.read_csv('submissionfile_sample.csv')
print(sample.shape)
print(sample.head())
print(sample.columns.tolist())

(24, 5)
                    id  hourly_pow         agg_pow        may_bill  \
0  2025-05-01 00:00:00         0.0  동일한 5월의 전체 누적량  동일한 5월 전체 전기요금   
1  2025-05-01 01:00:00         0.0  동일한 5월의 전체 누적량  동일한 5월 전체 전기요금   
2  2025-05-01 02:00:00         0.0  동일한 5월의 전체 누적량  동일한 5월 전체 전기요금   
3  2025-05-01 03:00:00         0.0             NaN               .   
4  2025-05-01 04:00:00         0.0             NaN               .   

         may_carbon  
0  동일한 5월의 전체 탄소 배출  
1  동일한 5월의 전체 탄소 배출  
2  동일한 5월의 전체 탄소 배출  
3               NaN  
4               NaN  
['id', 'hourly_pow', 'agg_pow', 'may_bill', 'may_carbon']


In [ ]:
train = hourly_full[hourly_full['dt'] < '2025-05-01'].copy()
test  = hourly_full[hourly_full['dt'] >= '2025-05-01'].copy()

print(f'학습 데이터: {train.shape}  ({train["dt"].min()} ~ {train["dt"].max()})')
print(f'테스트 데이터: {test.shape}  ({test["dt"].min()} ~ {test["dt"].max()})')

# 피처 정의 (EDA 근거)
features = [
    'hour', 'hour_sin', 'hour_cos',
    'currentR', 'currentS', 'currentT',
    'reactivePowerLagging'
]
target = 'hourly_pow'

X_train = train[features]
y_train = train[target]
X_test  = test[features]
y_test  = test[target]

print(f'\n피처: {features}')

## 3. LightGBM 베이스라인

In [ ]:
model_lgb = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)
model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

y_pred_lgb = model_lgb.predict(X_test)

def evaluate(y_true, y_pred, model_name):
    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    smape = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true))) * 100
    print(f'[{model_name}]')
    print(f'  MAE:   {mae:.2f}')
    print(f'  RMSE:  {rmse:.2f}')
    print(f'  SMAPE: {smape:.4f}%')
    return {'model': model_name, 'MAE': mae, 'RMSE': rmse, 'SMAPE': smape}

results = []
results.append(evaluate(y_test, y_pred_lgb, 'LightGBM'))

In [ ]:
# 피처 중요도
plt.figure(figsize=(10, 5))
lgb.plot_importance(model_lgb, max_num_features=10, importance_type='gain')
plt.title('LightGBM 피처 중요도')
plt.tight_layout()
plt.savefig('model_01_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 예측 시각화
plt.figure(figsize=(18, 5))
plt.plot(test['dt'], y_test.values, label='실제값', color='steelblue', linewidth=0.8)
plt.plot(test['dt'], y_pred_lgb, label='예측값', color='tomato', linewidth=0.8, alpha=0.8)
plt.title('LightGBM 예측 결과 (5월)')
plt.xlabel('날짜')
plt.ylabel('hourly_pow (kW)')
plt.legend()
plt.tight_layout()
plt.savefig('model_02_lgb_pred.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. lag 피처 추가 실험
> EDA에서 자기상관 없었지만 — 실제 성능 차이 확인

In [ ]:
# lag 피처 생성
hourly_full['lag_1']  = hourly_full['hourly_pow'].shift(1)
hourly_full['lag_24'] = hourly_full['hourly_pow'].shift(24)
hourly_full['lag_168'] = hourly_full['hourly_pow'].shift(168)
hourly_full['rolling_mean_24']  = hourly_full['hourly_pow'].shift(1).rolling(24).mean()
hourly_full['rolling_std_24']   = hourly_full['hourly_pow'].shift(1).rolling(24).std()

# lag 포함 학습/테스트 재분리
train_lag = hourly_full[hourly_full['dt'] < '2025-05-01'].dropna().copy()
test_lag  = hourly_full[hourly_full['dt'] >= '2025-05-01'].dropna().copy()

features_lag = features + ['lag_1', 'lag_24', 'lag_168', 'rolling_mean_24', 'rolling_std_24']

X_train_lag = train_lag[features_lag]
y_train_lag = train_lag[target]
X_test_lag  = test_lag[features_lag]
y_test_lag  = test_lag[target]

model_lgb_lag = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)
model_lgb_lag.fit(
    X_train_lag, y_train_lag,
    eval_set=[(X_test_lag, y_test_lag)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

y_pred_lag = model_lgb_lag.predict(X_test_lag)
results.append(evaluate(y_test_lag, y_pred_lag, 'LightGBM + lag'))

## 5. 이상탐지 — Rolling Z-score

In [ ]:
# Rolling Z-score (직전 24시간 기준)
hourly_full['rolling_mean'] = hourly_full['hourly_pow'].rolling(window=24, min_periods=1).mean()
hourly_full['rolling_std']  = hourly_full['hourly_pow'].rolling(window=24, min_periods=1).std()
hourly_full['z_score'] = (
    (hourly_full['hourly_pow'] - hourly_full['rolling_mean']) / hourly_full['rolling_std']
)

# 임계값 Z > 2.5 또는 Z < -2.5 → 이상
threshold = 2.5
hourly_full['is_anomaly'] = hourly_full['z_score'].abs() > threshold

anomalies = hourly_full[hourly_full['is_anomaly']]
print(f'이상 탐지 건수: {len(anomalies)}건 / 전체 {len(hourly_full)}시간')
print(anomalies[['dt', 'hourly_pow', 'z_score']].head(10))

In [ ]:
# 이상탐지 시각화
plt.figure(figsize=(18, 5))
plt.plot(hourly_full['dt'], hourly_full['hourly_pow'],
         color='steelblue', linewidth=0.6, label='hourly_pow')
plt.scatter(anomalies['dt'], anomalies['hourly_pow'],
            color='red', s=20, zorder=5, label=f'이상탐지 (Z>{threshold})')
plt.title('Rolling Z-score 이상탐지 결과')
plt.xlabel('날짜')
plt.ylabel('hourly_pow (kW)')
plt.legend()
plt.tight_layout()
plt.savefig('model_03_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 이상탐지 — Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest

iso_features = ['hourly_pow', 'currentR', 'currentS', 'currentT', 'reactivePowerLagging']
X_iso = hourly_full[iso_features].dropna()

iso_model = IsolationForest(
    contamination=0.01,  # 전체의 1%를 이상치로 가정
    random_state=42,
    n_estimators=100
)
hourly_full.loc[X_iso.index, 'iso_pred'] = iso_model.fit_predict(X_iso)

# -1 = 이상, 1 = 정상
iso_anomalies = hourly_full[hourly_full['iso_pred'] == -1]
print(f'Isolation Forest 이상 탐지: {len(iso_anomalies)}건')

plt.figure(figsize=(18, 5))
plt.plot(hourly_full['dt'], hourly_full['hourly_pow'],
         color='steelblue', linewidth=0.6, label='hourly_pow')
plt.scatter(iso_anomalies['dt'], iso_anomalies['hourly_pow'],
            color='orange', s=20, zorder=5, label='Isolation Forest 이상')
plt.title('Isolation Forest 이상탐지 결과')
plt.xlabel('날짜')
plt.ylabel('hourly_pow (kW)')
plt.legend()
plt.tight_layout()
plt.savefig('model_04_isolation_forest.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 모델 성능 비교 & 제출 파일 생성

In [ ]:
# 성능 비교표
results_df = pd.DataFrame(results)
print('=== 모델 성능 비교 ===')
print(results_df.to_string(index=False))

In [ ]:
# 제출 파일 생성
# 성능 더 좋은 모델 선택 (결과 보고 y_pred_lgb 또는 y_pred_lag로 변경)
best_pred = y_pred_lgb  # 결과 보고 교체

# agg_pow = 5월 전체 누적 합계
agg_pow = best_pred.sum()
may_bill   = agg_pow * 180
may_carbon = agg_pow * 0.424

print(f'agg_pow:    {agg_pow:,.2f} kWh')
print(f'may_bill:   {may_bill:,.0f} 원')
print(f'may_carbon: {may_carbon:,.2f} kgCO₂')

# 제출 파일 (샘플 파일 형식 맞춰서 수정 필요)
submission = test[['dt']].copy()
submission['hourly_pow'] = best_pred
submission['agg_pow'] = agg_pow  # 전 행 동일값
submission['may_bill'] = may_bill
submission['may_carbon'] = may_carbon

submission.to_csv('submission.csv', index=False)
print('\nsubmission.csv 저장 완료!')
print(submission.head())